# machine-learning_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (105).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 5 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install cvxpy xgboost

In [ ]:
import pandas as pd
import numpy as np
# Load data
ohlcv = pd.read_csv('/content/drive/MyDrive/3year_5min_data.csv')
ohlcv

In [ ]:
tradebook = pd.read_csv('/content/drive/MyDrive/tradebook_adanient.csv')
tradebook

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Convert dates to datetime format
ohlcv["Date"] = pd.to_datetime(ohlcv["Date"])
tradebook["Entry Time"] = pd.to_datetime(tradebook["Entry Time"])

# Extract Year
ohlcv["Year"] = ohlcv["Date"].dt.year
tradebook["Entry Year"] = tradebook["Entry Time"].dt.year

# Compute Holding Period Return (HPR) for each Ticker per year
yearly_returns = (
    ohlcv.groupby(["Ticker", "Year"])["close"]
    .agg(lambda x: (x.iloc[-1] - x.iloc[0]) / x.iloc[0] if x.iloc[0] != 0 else np.nan)
    .reset_index()
    .rename(columns={"close": "HPR"})
)

# Compute Strategy Returns from Tradebook per year
strategy_returns = (
    tradebook.groupby(["Ticker", "Entry Year"])["PnL"]
    .sum()
    .reset_index()
    .rename(columns={"PnL": "Strategy_Return", "Entry Year": "Year"})
)

# Merge Returns
data = yearly_returns.merge(strategy_returns, on=["Ticker", "Year"], how="left")
data.fillna(0, inplace=True)

# Feature Engineering: Compute Trade Statistics
summary = tradebook.groupby("Ticker").agg(
    total_pnl=("PnL", "sum"),
    total_trades=("PnL", "count"),
    win_rate=("PnL", lambda x: (x > 0).sum() / len(x) if len(x) > 0 else np.nan),
    max_drawdown=("Cumulative PnL", lambda x: (x - x.cummax()).min() if not x.empty else np.nan),
    profit_factor=("PnL", lambda x: x[x > 0].sum() / abs(x[x < 0].sum()) if abs(x[x < 0].sum()) > 0 else np.nan)
).reset_index()

# Merge Summary Statistics
data = data.merge(summary, on="Ticker", how="left").fillna(0)

# Clean Data
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

# Rolling Training and Prediction (Yearly)
predictions = []
years = sorted(data["Year"].unique())

top_stocks_per_year = {}

for i in range(2, len(years)):  # Start from 3rd year to have training data
    train_years = years[:i]
    test_year = years[i]

    train_data = data[data["Year"].isin(train_years)]
    test_data = data[data["Year"] == test_year]

    X_train = train_data.drop(columns=["Ticker", "Year", "Strategy_Return"])
    y_train = train_data["Strategy_Return"]

    X_test = test_data.drop(columns=["Ticker", "Year", "Strategy_Return"])
    tickers_test = test_data["Ticker"]

    # Train Random Forest
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predict Strategy Returns
    predicted_returns = model.predict(X_test)
    test_data["Predicted_Strategy_Return"] = predicted_returns

    predictions.append(test_data)

    # Top 10 stocks
    top_10_tickers = test_data.nlargest(10, "Predicted_Strategy_Return")["Ticker"].tolist()
    top_stocks_per_year[f"{test_year}"] = top_10_tickers

    print(f"\nTop 10 Predicted Stocks for {test_year}:")
    print(top_10_tickers)

# Concatenate all predictions
final_predictions = pd.concat(predictions)

# Save to CSV
final_predictions.to_csv("yearly_predictions.csv", index=False)

# Print all yearly top stocks
print("\nFinal Walk-Forward Optimization Results (Yearly):")
for year, tickers in top_stocks_per_year.items():
    print(f"{year}: {tickers}")
